# VLM second check: batch pass on AMD

Runs Qwen2.5-VL-7B-Instruct with vLLM's offline `LLM` class over a folder of crack crops and writes one JSON line
per crop to `results/vlm-<date>.jsonl`. The reply is constrained to `holdpoint.vlm.SCHEMA` (structured outputs)
and validated again with `holdpoint.vlm.parse_reply` (closed vocabulary, no numeric content).

Meant for an AMD Instinct GPU with ROCm PyTorch and vLLM. Re-running skips crops already in the output file.

In [ ]:
import os, sys, json, time, hashlib, platform, subprocess
from datetime import datetime, timezone
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import torch
print("torch", torch.__version__, "| HIP", getattr(torch.version, "hip", None))
DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"  # ROCm uses the torch.cuda API
print("device:", DEVICE)
try:
    print(subprocess.run(["amd-smi", "static", "--asic"], capture_output=True, text=True, timeout=30).stdout[:2000])
except FileNotFoundError:
    print("amd-smi not found")

In [ ]:
CROPS_DIR = Path(os.environ.get("HOLDPOINT_CROPS", REPO / "data" / "crops"))
MODEL = os.environ.get("HOLDPOINT_VLLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct")
BATCH = int(os.environ.get("HOLDPOINT_VLM_BATCH", "32"))
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d")
OUT = REPO / "results" / f"vlm-{STAMP}.jsonl"
OUT.parent.mkdir(parents=True, exist_ok=True)
COMMAND = f"notebooks/vlm_batch.ipynb (CROPS_DIR={CROPS_DIR.name}, MODEL={MODEL}, BATCH={BATCH})"

files = sorted(p for p in CROPS_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
done = set()
if OUT.exists():
    done = {json.loads(l)["file"] for l in OUT.read_text().splitlines() if l.strip()}
todo = [p for p in files if str(p.relative_to(CROPS_DIR)) not in done]
print(f"{len(files)} crops, {len(done)} already done, {len(todo)} to run -> {OUT}")

In [ ]:
import vllm
from vllm import LLM, SamplingParams
from holdpoint.vlm import SCHEMA, messages_for, parse_reply, png_data_url
from holdpoint.backends import image_sha256

def sampling_params():
    # structured-output API name changed across vLLM versions
    try:
        from vllm.sampling_params import StructuredOutputsParams
        return SamplingParams(temperature=0, max_tokens=120, structured_outputs=StructuredOutputsParams(json=SCHEMA))
    except ImportError:
        from vllm.sampling_params import GuidedDecodingParams
        return SamplingParams(temperature=0, max_tokens=120, guided_decoding=GuidedDecodingParams(json=SCHEMA))

llm = LLM(model=MODEL, max_model_len=4096, limit_mm_per_prompt={"image": 1})
SP = sampling_params()
print("vllm", vllm.__version__)

In [ ]:
import cv2

def load_rgb(p):
    bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"cannot read {p}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

t0 = time.time()
n = 0
with OUT.open("a") as fh:
    for i in range(0, len(todo), BATCH):
        chunk = todo[i : i + BATCH]
        imgs = [load_rgb(p) for p in chunk]
        outs = llm.chat([messages_for(png_data_url(im)) for im in imgs], SP, use_tqdm=False)
        for p, im, o in zip(chunk, imgs, outs):
            raw = o.outputs[0].text
            r = parse_reply(raw)
            fh.write(json.dumps({
                "file": str(p.relative_to(CROPS_DIR)),
                "file_sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
                "image_sha256": image_sha256(im),
                "raw": raw,
                "label": r.label,
                "sentence": r.sentence,
                "flags": r.flags,
                "provenance": {
                    "backend": "amd_notebook",
                    "device": DEVICE,
                    "model": MODEL,
                    "vllm": vllm.__version__,
                    "timestamp": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
                    "command": COMMAND,
                },
            }) + "\n")
        fh.flush()
        n += len(chunk)
        print(f"{n}/{len(todo)}  {n / (time.time() - t0):.2f} img/s")

In [ ]:
from collections import Counter
rows = [json.loads(l) for l in OUT.read_text().splitlines() if l.strip()]
print(len(rows), "rows")
print("labels:", Counter(r["label"] for r in rows).most_common())
print("flags:", Counter(f for r in rows for f in r["flags"]).most_common())